# Diagnostic: Are favorites really priced at 55c with 50 seconds left?

Checks whether the grid search results make sense by looking at the raw tick data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

TICK_CSV = "/content/drive/MyDrive/commod15min_ticks.csv"
df = pd.read_csv(TICK_CSV, usecols=["ticker", "asset", "yes_mid", "sec_to_close"])
df = df.dropna(subset=["yes_mid", "sec_to_close"])
df["minute_in_cycle"] = 15.0 - df["sec_to_close"] / 60.0
df = df[(df["minute_in_cycle"] >= 0) & (df["minute_in_cycle"] <= 15)]
df["fav_price"] = np.maximum(df["yes_mid"], 100.0 - df["yes_mid"])
print(f"Total ticks: {len(df):,}")
print(f"Total unique windows (tickers): {df['ticker'].nunique():,}")
print(f"Assets: {sorted(df['asset'].unique())}")

In [ ]:
# CELL 3: Total windows per asset
windows_per_asset = df.groupby('asset')['ticker'].nunique().sort_values(ascending=False)
print("WINDOWS PER ASSET:")
print("=" * 40)
for asset, count in windows_per_asset.items():
    print(f"  {asset:<12} {count:>6,} windows")
print(f"  {'TOTAL':<12} {windows_per_asset.sum():>6,} windows")

In [ ]:
# CELL 4: What does the favorite price distribution look like in the last 50 seconds?
last_50s = df[df["sec_to_close"] <= 50].copy()
print(f"Ticks in last 50 seconds: {len(last_50s):,}")
print(f"Unique windows with ticks in last 50s: {last_50s['ticker'].nunique():,}")
print()
print("FAVORITE PRICE DISTRIBUTION (last 50s, all assets):")
print("=" * 50)
bins = [50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 101]
labels = ['50-55', '55-60', '60-65', '65-70', '70-75', '75-80', '80-85', '85-90', '90-95', '95-100']
last_50s['fav_bin'] = pd.cut(last_50s['fav_price'], bins=bins, labels=labels, right=False)
dist = last_50s['fav_bin'].value_counts().sort_index()
total = len(last_50s)
for b, count in dist.items():
    pct = 100 * count / total
    bar = '#' * int(pct)
    print(f"  {b:>8}c : {count:>8,} ticks ({pct:>5.1f}%) {bar}")

In [ ]:
# CELL 5: Same distribution but PER ASSET
print("FAVORITE PRICE DISTRIBUTION (last 50s) PER ASSET:")
print("=" * 70)
for asset in sorted(last_50s['asset'].unique()):
    sub = last_50s[last_50s['asset'] == asset]
    n_windows = sub['ticker'].nunique()
    print(f"\n  {asset} ({n_windows} windows with data in last 50s):")
    d = sub['fav_bin'].value_counts().sort_index()
    t = len(sub)
    for b, count in d.items():
        pct = 100 * count / t
        print(f"    {b:>8}c : {count:>6,} ticks ({pct:>5.1f}%)")

In [ ]:
# CELL 6: The critical question -- how many WINDOWS have ANY tick with
# favorite in [55, 60) during the last 50 seconds?
# This is what the grid search counts as a potential entry.
print("WINDOWS WITH AT LEAST ONE TICK where fav is in [X, X+5) in last 50s:")
print("=" * 70)
print(f"{'Band':>10} {'Windows':>10} {'% of all':>10}   per asset...")
print("-" * 70)

all_windows_50s = last_50s['ticker'].nunique()
for lo in range(51, 91):
    hi = lo + 5
    mask = (last_50s['fav_price'] >= lo) & (last_50s['fav_price'] <= hi)
    if not mask.any():
        if lo <= 60:  # only print the interesting range
            print(f"  {lo}-{hi}c  :          0 (  0.0%)")
        continue
    hit_tickers = last_50s[mask]['ticker'].nunique()
    pct = 100 * hit_tickers / all_windows_50s
    
    # per-asset breakdown
    asset_str = ""
    if lo <= 65:  # detailed for the suspicious range
        for a in sorted(last_50s['asset'].unique()):
            sub = last_50s[last_50s['asset'] == a]
            a_mask = (sub['fav_price'] >= lo) & (sub['fav_price'] <= hi)
            a_hits = sub[a_mask]['ticker'].nunique()
            if a_hits > 0:
                asset_str += f" {a}:{a_hits}"
    
    print(f"  {lo}-{hi}c  : {hit_tickers:>8,} ({pct:>5.1f}%){asset_str}")

In [ ]:
# CELL 7: Show 20 sample entry ticks for the top combo (55-60c, last 50s)
# So we can see the actual yes_mid values
print("SAMPLE ENTRY TICKS: favorite 55-60c in last 50 seconds")
print("=" * 80)
print(f"{'ticker':<30} {'asset':<10} {'yes_mid':>8} {'fav_price':>10} {'sec_to_close':>13}")
print("-" * 80)

band_mask = (last_50s['fav_price'] >= 55) & (last_50s['fav_price'] <= 60)
band_ticks = last_50s[band_mask]

# Get the FIRST tick per window (what the grid search would use as entry)
first_per_window = band_ticks.sort_values('minute_in_cycle').groupby('ticker').first().reset_index()
sample = first_per_window.head(20)
for _, r in sample.iterrows():
    print(f"  {r['ticker']:<28} {r['asset']:<10} {r['yes_mid']:>8.1f} {r['fav_price']:>10.1f} {r['sec_to_close']:>13.1f}")

print(f"\nTotal windows with entry in 55-60c band, last 50s: {len(first_per_window)}")